# Boundary Equilibrium Generative Adversarial Network (BEGAN) Applied to CelebA Dataset

Antonio Esteves @ UMinho, Jun 2024

## Import the necessary libraries

In [ ]:
import os
import numpy as np
import math
import time
import wandb
import yaml
import random
from   pathlib                import Path
from   PIL                    import Image
import matplotlib.pyplot      as     plt
from   natsort                import natsorted

import torch
import torch.nn               as     nn
import torch.nn.functional    as     F
import torchvision.transforms as     transforms
from   torchvision.utils      import save_image, make_grid
from   torch.utils.data       import DataLoader, Dataset
from   torchvision            import datasets
from   torch.autograd         import Variable

from   tqdm.notebook          import trange, tqdm

## Read the configuration file

In [ ]:
LOAD_TRAINED_MODEL     = False
SKIP_TRAIN_MODEL       = False

CONFIG_FILE = '../config/began_celeba_128x128_05.yaml'

with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

In [ ]:
print('parameters:')
for key, value in config.items():
    print(f'\t{key}: {value}')

## Seed everything

Let us seed everything to make the results reproducible.

In [ ]:
def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(config['manual_seed'])

## Initializations and create necessary folders

In [ ]:
train_dir= Path(config["dataset_path"])

print(torch.__version__)

# Setup device agnostic code

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f'Using {device} for computing')

# Location where we will save here the images generated during LSGAN training
RESULTS_PATH=f'results/{config["experiment_name"]}'
os.makedirs(RESULTS_PATH, exist_ok=True)

# Location where the trained models will be saved
MODELS_PATH      = os.path.join(os.getcwd(), 'models')
os.makedirs(MODELS_PATH, exist_ok=True)

## Login into Weights & Bias

In [ ]:
wandb.login()

## Track metadata and hyperparameters with Weights & Bias

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = config

wandb.init(
    project = 'OUR_WANDB_PROJECT_ID',
    entity  = 'OUR_WANDB_ENTITY', 
    config  = config_wandb
)

## Exploring the CelebA dataset

In [ ]:
# Get all training image paths
train_image_list = list(train_dir.glob("*.jpg"))

print(f'Training set size: {len(train_image_list)}')

In [ ]:
# Visualizing random images

# Pick a random image path

rand_image_path = random.choice(train_image_list)
print(f'Randomly selected image:\n\t{rand_image_path}')

# Open the image using pillow library

img = Image.open(rand_image_path)

# Show the image and print image metadata

print(f'Random image path:   {rand_image_path}')
print(f'Random image height: {img.height}')
print(f'Random image width:  {img.width}')

display(img)

In [ ]:
# Visualize an image using matplotlib

# convert the image 'img' to a numpy array

img_array = np.asarray(img)

# plot the image with matplotlib

plt.figure(figsize=(10,7))
plt.imshow(img_array)
plt.title(f'Image shape: {img_array.shape} (height,width,channels)')
plt.axis(False)

## Create a custom Dataset from the images in a folder and the associated attributes using `torch.utils.data.Dataset`

In [ ]:
class CustomDataSet(Dataset):

    def __init__(self, root_dir, transform):
        self.root_dir     = root_dir
        self.transform    = transform
        self.all_images   = os.listdir(root_dir)
        self.total_images = natsorted(self.all_images)

    def __len__(self):
        return len(self.total_images)

    def __getitem__(self, idx):
        img_loc      = os.path.join(self.root_dir, self.total_images[idx])
        image        = Image.open(img_loc).convert("RGB")
        tensor_image = self.transform(image)
        return tensor_image

## i. Define the transformation that will be applied to the images

- resize the images to the resolution that we want,
- convert the images to tensors,
- apply some augmentations,

## ii. Instantiate a Custom Dataset

## iii. Create a training DataLoader

- Select the current batch size from the list `batch_sizes`, using as index `int(log2(image_size/4)`.
- This is actually how we implement a mini-batch size that depends on the images resolution.

In [ ]:
def get_loader(image_size, batch_size, dataset_train_dir):

    train_transform = transforms.Compose(
        [
        transforms.Resize(config['image_size']),
        transforms.CenterCrop(config['image_size']),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
    )

    train_data = CustomDataSet(
        root_dir  = dataset_train_dir,
        transform = train_transform,
    )

    train_loader = DataLoader(
        dataset     = train_data,
        batch_size  = batch_size,
        shuffle     = True,
        drop_last   = True,
        num_workers = 4,
        pin_memory  = True,
    )

    return train_loader, train_data

Let us check if everything works fine and display a few real images.

In [ ]:
def check_dataloader(image_size, batch_size, dataset_train_dir):
    NR, NC    = 3, 3
    loader, _ = get_loader(image_size, batch_size, dataset_train_dir)
    imgs      = next(iter(loader))
    print(f'Batch of images shape: {imgs.shape}')   # BS, Ch, H, W

    if NR*NC > imgs.shape[0]:
        NR = 2
        if NR*NC > imgs.shape[0]:
            NR = 1
            if NR*NC > imgs.shape[0]:
                NC = 2

    _, ax    = plt.subplots(NR, NC, figsize=(3*NC,3*NR))
    plt.suptitle(
        'Some real images of {config["dataset"]} dataset',
        fontsize=15,
        fontweight='bold'
    )

    index = 0
    for r in range(NR):
        for c in range(NC):
            index += 1
            if NR==1:
                ax[c].imshow((imgs[index].permute(1,2,0)+1)/2) 
            else:
                ax[r][c].imshow((imgs[index].permute(1,2,0)+1)/2) 

In [ ]:
check_dataloader(
    image_size        = config["image_size"], 
    batch_size        = config["batch_size"], 
    dataset_train_dir = train_dir
)

## BEGAN model implementation

In [ ]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels,in_channels,kernel_size=3,stride=1,padding=1),
        nn.ELU(True),
        nn.Conv2d(in_channels,in_channels,kernel_size=3,stride=1,padding=1),
        nn.ELU(True),
        nn.Conv2d(in_channels,out_channels,kernel_size=1,stride=1,padding=0),
        nn.AvgPool2d(kernel_size=2,stride=2)
    )

def deconv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels,out_channels,kernel_size=3,stride=1,padding=1),
        nn.ELU(True),
        nn.Conv2d(out_channels,out_channels,kernel_size=3,stride=1,padding=1),
        nn.ELU(True),
        nn.UpsamplingNearest2d(scale_factor=2)
    )

def weights_init_normal(m):
    '''
    Custom weight initialization for "Conv" and "BatchNorm" layers.
    '''
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        torch.nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        torch.nn.init.normal_(m.weight.data, 1.0, 0.02)
        torch.nn.init.constant_(m.bias.data, 0.0)

### Generator model

In [ ]:
class Generator(nn.Module):
    '''
    Generator model of BEGAN.
    It is designed for 64x64 or 128x128 images only.
    For other shape images, the architecture has to be adjusted.
    '''

    def __init__(self, channels, ngf, nz, imageSize):
        super(Generator,self).__init__()
        
        self.embed1 = nn.Linear(nz, ngf*8*8)
        # output shape: 8 x 8 x ngf
        
        self.deconv1 = deconv_block(ngf, ngf)
        # output shape: 16 x 16 x ngf
        
        self.deconv2 = deconv_block(ngf, ngf)
        # output shape: 32 x 32 x ngf

        self.deconv3 = deconv_block(ngf, ngf)
        # output shape: 64 x 64 x ngf

        if(imageSize == 128):
            self.deconv4 = deconv_block(ngf, ngf)
            # output shape: 128 x 128 x ngf

            self.deconv5 = nn.Sequential(
                nn.Conv2d(ngf, ngf, kernel_size=3, stride=1, padding=1),
                nn.ELU(True),
                nn.Conv2d(ngf, ngf, kernel_size=3, stride=1, padding=1),
                nn.ELU(True),
                nn.Conv2d(ngf, channels, kernel_size=3, stride=1, padding=1)
            )
        else:
            self.deconv4 = nn.Sequential(
                nn.Conv2d(ngf, ngf, kernel_size=3, stride=1, adding=1),
                nn.ELU(True),
                nn.Conv2d(ngf, ngf, kernel_size=3, stride=1, padding=1),
                nn.ELU(True),
                nn.Conv2d(ngf, channels, kernel_size=3, stride=1, padding=1),
                nn.Tanh()
            )
        self.ngf       = ngf
        self.imageSize = imageSize

    def forward(self,x):
        out = self.embed1(x)
        out = out.view(out.size(0), self.ngf, 8, 8)
        out = self.deconv1(out)
        out = self.deconv2(out)
        out = self.deconv3(out)
        out = self.deconv4(out)
        if(self.imageSize == 128):
            out = self.deconv5(out)
        return out


### Discriminator model

In [ ]:
class Discriminator(nn.Module):

    def __init__(self, channels, ndf, hidden_size, imageSize):
        super(Discriminator,self).__init__()

        # Encoder module ........................................................

        # input shape: image_size x image_size x channels
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(channels, ndf, kernel_size=3, stride=1, padding=1),
            nn.ELU(True),
            conv_block(ndf, ndf)
        )
        # output shape: image_size/2 x image_size/2 x ndf

        self.conv2 = conv_block(ndf, ndf*2)
        # output shape: image_size/4 x image_size/4 x ndf*2

        self.conv3 = conv_block(ndf*2, ndf*3)
        # output shape: image_size/8 x image_size/8 x ndf*3

        if(imageSize == 64):
            self.conv4 = nn.Sequential(
                nn.Conv2d(ndf*3,ndf*3,kernel_size=3,stride=1,padding=1),
                nn.ELU(True),
                nn.Conv2d(ndf*3,ndf*3,kernel_size=3,stride=1,padding=1),
                nn.ELU(True)
            )
            # output shape: image_size/8 x image_size/8 x ndf*3 = 8 x 8 x ndf*3
            self.embed1 = nn.Linear(ndf*3*8*8, hidden_size)
            # output shape: hidden_size
        else:
            self.conv4 = conv_block(ndf*3, ndf*4)
            # output shape: image_size/16 x image_size/16 x ndf*4

            self.conv5 = nn.Sequential(
                nn.Conv2d(ndf*4,ndf*4,kernel_size=3,stride=1,padding=1),
                nn.ELU(True),
                nn.Conv2d(ndf*4,ndf*4,kernel_size=3,stride=1,padding=1),
                nn.ELU(True)
            )
            # output shape: image_size/16 x image_size/16 x ndf*4 = 8 x 8 x ndf*4
            self.embed1 = nn.Linear(ndf*4*8*8, hidden_size)
            # output shape: hidden_size

        # Decoder module ........................................................

        self.embed2 = nn.Linear(hidden_size, ndf*8*8)
        # output shape: 8 x 8 x ndf

        self.deconv1 = deconv_block(ndf, ndf)
        # output shape: 16 x 16 x ndf

        self.deconv2 = deconv_block(ndf, ndf)
        # output shape: 32 x 32 x ndf
        
        self.deconv3 = deconv_block(ndf, ndf)
        # output shape: 64 x 64 x ndf

        if(imageSize == 64):
            self.deconv4 = nn.Sequential(
                nn.Conv2d(ndf, ndf, kernel_size=3, stride=1, padding=1),
                nn.ELU(True),
                nn.Conv2d(ndf, ndf, kernel_size=3, stride=1, padding=1),
                nn.ELU(True),
                nn.Conv2d(ndf, channels, kernel_size=3, stride=1, padding=1)
            )
            # output shape: 64 x 64 x ndf
        else:
            self.deconv4 = deconv_block(ndf, ndf)
            # output shape: 128 x 128 x ndf

            self.deconv5 = nn.Sequential(
                nn.Conv2d(ndf, ndf, kernel_size=3, stride=1, padding=1),
                nn.ELU(True),
                nn.Conv2d(ndf, ndf, kernel_size=3, stride=1, padding=1),
                nn.ELU(True),
                nn.Conv2d(ndf, channels, kernel_size=3, stride=1, padding=1),
                nn.Tanh()
            )
            # output shape: 128 x 128 x ndf

        self.ndf       = ndf
        self.imageSize = imageSize

    def forward(self,x):
        # Encoder module ..................................
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        out = self.conv4(out)
        if(self.imageSize == 128):
            out = self.conv5(out)
            out = out.view(out.size(0), self.ndf*4 * 8 * 8)
        else:
            out = out.view(out.size(0), self.ndf*3 * 8 * 8)
        out = self.embed1(out)
        
        # Decoder module ..................................
        out = self.embed2(out)
        out = out.view(out.size(0), self.ndf, 8, 8)
        out = self.deconv1(out)
        out = self.deconv2(out)
        out = self.deconv3(out)
        out = self.deconv4(out)
        if(self.imageSize == 128):
            out = self.deconv5(out)
        return out

## Utility functions

In [ ]:
def time_format(seconds: int) -> str:
    if seconds is not None:
        seconds = int(seconds)
        d = seconds // (3600 * 24)
        h = seconds // 3600 % 24
        m = seconds % 3600 // 60
        s = seconds % 3600 % 60
        if d > 0:
            return '{:02d}D {:02d}H {:02d}m {:02d}s'.format(d, h, m, s)
        elif h > 0:
            return '{:02d}H {:02d}m {:02d}s'.format(h, m, s)
        elif m > 0:
            return '{:02d}m {:02d}s'.format(m, s)
        elif s > 0:
            return '{:02d}s'.format(s)
    return '-'


## Functions to save and load the models to/from file

In [ ]:
def save_model_and_results(discriminator, generator, results, hyperparameters, file_name):
    results_to_save = {
        'discriminator':   discriminator.state_dict(),
        'generator':       generator.state_dict(),
        'results':         results,
        'hyperparameters': hyperparameters,
    }

    torch.save(
        results_to_save,
        file_name,
    )

In [ ]:
def load_model(discriminator, generator, file_name, device):
    '''
    Given instances of the generator and discriminator models, loads from file 'file_name':
    (i)   the weights of both models,
    (ii)  the results obtained during model training and
    (iii) the training hyperparameters used to train the models,
    and put the models on 'device'.

    Returns the loaded results and the loaded hyperparameters.
    '''

    results_loaded = torch.load(file_name)

    discriminator.load_state_dict(results_loaded['discriminator'])
    discriminator.to(device)

    generator.load_state_dict(results_loaded['generator'])
    generator.to(device)

    # Returns the saved results and the saved hyperparameters
    return results_loaded['results'], results_loaded['hyperparameters']

## Training the model

In [ ]:
# Instantiate the discriminator and generator
discriminator     = Discriminator(
    config['channels'],
    config['disc_channels'],
    config['disc_hidden_size'],
    config['image_size'],
    ).train().to(device)

generator         = Generator(
    config['channels'],
    config['gen_channels'],
    config['z_dim'],
    config['image_size'],
    ).train().to(device)

# Initialize the weights
discriminator.apply(weights_init_normal)
generator.apply(weights_init_normal)

# Select the optimizers
optimizer_D = torch.optim.Adam(
    discriminator.parameters(),
    lr    = config["lr"],
    betas = (config["beta1"], config["beta2"])
)
optimizer_G = torch.optim.Adam(
    generator.parameters(),
    lr    = config["lr"],
    betas = (config["beta1"], config["beta2"])
)

### Print the generator model summary

In [ ]:
from torchinfo import summary

aux_data = torch.randn(config['batch_size'], config["z_dim"], device=device)
summary(
    generator,
    input_data   = aux_data,
    col_width    = 16,
    col_names    = ["kernel_size", "output_size", "num_params"],
    row_settings = ["var_names"],
)

### Print the discriminator model summary

In [ ]:
from torchinfo import summary

aux_data = torch.randn(
    (
    config['batch_size'],
    config['channels'], 
    config['image_size'], 
    config['image_size']
    )).to(device)

summary(
    discriminator,
    input_data   = aux_data,
    col_width    = 16,
    col_names    = ["kernel_size", "output_size", "num_params"],
    row_settings = ["var_names"],
)

In [ ]:
# Create an empty dictionary to store the training results .................
results = {
    'd_loss':      [],
    'g_loss':      [],
    'k':           [],
    'convergence': [],
    'epoch_training_time': 0.0,
}

# Instantiate the training dataloader ......................................

loader, _ = get_loader(config["image_size"], config["batch_size"], train_dir)

In [ ]:
def adjust_learning_rate(optimizer, epoch, config):
    """
    Decreases the learning rate every config['lr_decay_interval'] epochs.
    When the lower bound config['lr_lower_bound'] is reached, it is set
    the initial learning rate value.
    """
    lr = config['lr'] * (0.95 ** (epoch // config['lr_decay_interval']))
    if lr < config['lr_lower_bound']:
        lr = config['lr']
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    return optimizer

def train_began(
        discriminator,
        generator,
        optimizer_D,
        optimizer_G,
        dataloader,
        config,
        results,
        device,
    ):

    # For logging purposes
    ref_batch_size = 25
    img_cnt        = 0

    # Fixed input batch of random vectors to used to evaluate the generation 
    # quality during training
    ref_noise_batch = torch.randn(ref_batch_size, config["z_dim"], device=device)

    # -----------------------------
    #  Training loop
    # -----------------------------

    k = 0
    for epoch in tqdm(range(config['epochs']), desc="Epoch: "):

        ts = time.time()

        for i, real_imgs in enumerate(tqdm(dataloader)):

            real_imgs  = real_imgs.to(device)

            # Sample the noise to use as generator input
            z_in     = torch.randn(
                config['batch_size'],
                config["z_dim"],
                device=device,
            )

            # Generate a batch of images
            fake_imgs = generator(z_in)

            # -------------------------
            #  Train the discriminator
            # -------------------------

            optimizer_D.zero_grad()

            # Loss measures discriminator capacity to distinguish real 
            # from generated samples
            real_recons = discriminator(real_imgs)
            fake_recons = discriminator(fake_imgs.detach()) 

            real_error = torch.mean(torch.abs(real_recons-real_imgs))
            fake_error = torch.mean(torch.abs(fake_recons-fake_imgs.detach()))

            d_loss = real_error - k * fake_error
            d_loss.backward()
            optimizer_D.step()

            # ---------------------
            #  Train the generator
            # ---------------------

            optimizer_G.zero_grad()
           
            # Loss measures the generator capacity to trick the discriminator........
            fake_recons = discriminator(fake_imgs)
            g_loss      = torch.mean(torch.abs(fake_recons-fake_imgs))

            g_loss.backward()
            optimizer_G.step()

            # Calculate k, balance and convergence ..................................
            
            balance     = (config['gamma'] * real_error - fake_error).item()
            k           = min(max(k + config['lambda_k'] * balance,0),1)
            convergence = real_error.item() + np.abs(balance)

            # Save the results in a dictionary ......................................
            results["d_loss"].append(d_loss.item())
            results["g_loss"].append(g_loss.item())
            results["k"].append(k)
            results["convergence"].append(convergence)

            # Print progress metrics and save them to W&B ..........................
            if i % config["log_interval"] == 0:

                mean_d_loss      = np.mean(results["d_loss"][-config["log_interval"]:])
                mean_g_loss      = np.mean(results["g_loss"][-config["log_interval"]:])
                mean_k           = np.mean(results["k"][-config["log_interval"]:])
                mean_convergence = np.mean(results["convergence"][-config["log_interval"]:])

                print(f'epoch|iter: {epoch+1 :4d} | {i :5d} / {len(dataloader) :6d}', end = "  ")
                print(f'({(i*100)/len(dataloader) :0>5.1f}%)', end="  ")
                print(f'D loss: {mean_d_loss :0>10.7f}', end="  ")
                print(f'G loss: {mean_g_loss :0>10.7f}', end="  ")
                print(f'K: {mean_k :0>7.5f}', end="  ")
                print(f'Convg: {mean_convergence :.5f}', end="  ")
                print(f'LR: {optimizer_D.param_groups[0]["lr"] :.6f}')

                try:
                    # Log metrics to Weights & Biases ............................
                    wandb.log(
                        {
                        "discriminator_loss": mean_d_loss,
                        "generator_loss":     mean_g_loss,
                        "k":                  mean_k,
                        "convergence":        mean_convergence,
                        "epoch":              epoch+1,
                        }
                    )
                except Exception as ex:
                    print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

        # Apply the learning rate decay  ........................................

        optimizer_D = adjust_learning_rate(optimizer_D, epoch, config)
        optimizer_G = adjust_learning_rate(optimizer_G, epoch, config)

        # Save intermediate generated images ....................................

        if epoch % config["sampling_interval"] == 0:
            with torch.no_grad():
                log_generated_images = generator(ref_noise_batch)
                log_generated_images_resized = nn.Upsample(
                    scale_factor = 2, 
                    mode         = 'nearest',
                )(log_generated_images)

                out_path = os.path.join(
                    RESULTS_PATH,
                    f'{config["experiment_name"]}_generated_from_refZ_{str(img_cnt).zfill(6)}.jpg'
                )
                save_image(
                    log_generated_images_resized,
                    out_path,
                    nrow      = int(np.sqrt(ref_batch_size)),
                    normalize = True,
                )

            img_cnt += 1

        te        = time.time()
        texec_sec = te - ts
        texec_str = time_format(texec_sec)
        print(f'Epoch training time: {texec_str}')

        try:
            wandb.log(
                {
                "epoch_training_time_sec": texec_sec,
                }
            )
        except Exception as ex:
            print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

        ########################### UNCOMMENT THIS BLOCK OF CODE #################################
        #
        # Save the generator and discriminator in the 'models' directory   
        #if (epoch % config["checkp_interval"] == 0) or (epoch == config['epochs']-1):
        #    file_save_model = f'models/{config["experiment_name"]}.pth'
        #    save_model_and_results(
        #        discriminator,
        #        generator,
        #        results,
        #        config,
        #        file_save_model,
        #    )

        # ALTERNATIVE CODE (begin) ###############################################################
        if (epoch <= 40): 
            config["checkp_interval"] = 10
        else:
            config["checkp_interval"] = 1
        
        if (epoch % config["checkp_interval"] == 0) or (epoch == config['epochs']-1):
            file_save_model = f'models/checkpoints/{config["experiment_name"]}_{str(epoch).zfill(3)}.pth'
            save_model_and_results(
                discriminator,
                generator,
                results,
                config,
                file_save_model,
            )
        # ALTERNATIVE CODE (end) #################################################################


In [ ]:
# =========================================================================
# Train the model from the beginning
# =========================================================================

if LOAD_TRAINED_MODEL == False and SKIP_TRAIN_MODEL == False:

    train_began(
        discriminator,
        generator,
        optimizer_D,
        optimizer_G,
        loader,
        config,
        results,
        device,
    )

# =========================================================================
# Load the saved models
# =========================================================================

elif LOAD_TRAINED_MODEL == True:

    file_save_model = f'models/{config["experiment_name"]}.pth'
    results, _      = load_model(
        discriminator,
        generator,
        file_save_model,
        device,
    )

    # ---------------------------------------------------------------------
    # Continue training of the loaded models
    # ---------------------------------------------------------------------

    if SKIP_TRAIN_MODEL == False:

        train_began(
            discriminator,
            generator,
            optimizer_D,
            optimizer_G,
            loader,
            config,
            results,
            device,
        )

### Real Images vs. Fake Images

In [ ]:
if SKIP_TRAIN_MODEL == False:

    # Draw a batch of real images from the dataloader
    real_batch = next(iter(loader))

    # Generate a set of latent vectors
    noise = torch.randn(config['batch_size'], config["z_dim"], device=device)

    # Generate a set of fake images with G
    fake_batch = generator(noise).detach().cpu()

    real_grid = make_grid(
        real_batch.to(device)[:64],
        padding   = 5,
        normalize = True,
    ).cpu()

    fake_grid = make_grid(
        fake_batch.to(device)[:64],
        padding   = 5,
        normalize = True,
    ).cpu()

    # Plot the real images
    plt.figure(figsize=(15, 15))
    plt.subplot(1, 2, 1)
    plt.axis("off")
    plt.title("Real Images")
    plt.imshow(
        np.transpose(
            real_grid,
            (1, 2, 0),
        )
    )

    # Plot the fake images
    plt.subplot(1, 2, 2)
    plt.axis("off")
    plt.title("Fake Images")
    plt.imshow(np.transpose(fake_grid, (1, 2, 0)))
    plt.savefig(f'results/{config["experiment_name"]}/{config["experiment_name"]}_real_vs_generated.png')
    plt.show()

### Generate grids of images with the fully trained generator

In [ ]:
def generate_grid_images(generator, num_grids, grid_size, config, device):

    assert config["batch_size"] >= grid_size, f'Grid size must be less or equal to batch size={config["batch_size"]}'
    
    generator.eval()

    with torch.inference_mode():

        for num in range(num_grids):

            # Generate a set of latent vectors
            noise = torch.randn(config['batch_size'], config['z_dim'], device=device)

            # Generate a set of fake images with G
            fake = generator(noise).detach().cpu()
            
            if(config['batch_size'] > grid_size):
                fake = fake[:grid_size]

            # Create a grid with the generated images
            grid = make_grid(fake, padding=2, normalize=True)
            grid = grid.permute(1, 2, 0)
            grid = grid.numpy()

            # Display the grid of images
            _ = plt.figure(figsize=(10, 10), constrained_layout=True)
            plt.imshow(grid)

            # Save the grid of images as a PNG file
            file_png = f'results/{config["experiment_name"]}/{config["experiment_name"]}_generated_final_{str(num+1).zfill(3)}.png'
            plt.imsave(file_png, grid)

In [ ]:
generate_grid_images(generator, 16, 64, config, device)

In [ ]:
# Mark the Weights & Bias run as finished
wandb.finish()